# 02 · Reducción de dimensionalidad y clustering
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/es/02_dimensionality_reduction_clustering.ipynb)

**De Células a Clústeres: Introducción al Análisis de Single-Cell RNA-seq en Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Perú · 9 de noviembre de 2026  
Instructora: Dra. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Tiempo en el tutorial:** 10:15–10:45 (30 min)

**Al final de este notebook podrás:**
- Ejecutar PCA y decidir cuántos componentes principales conservar
- Construir un grafo de k vecinos más cercanos y un embedding UMAP (y saber lo que UMAP *no* te dice)
- Agrupar células con el algoritmo de Leiden y explorar el efecto de la resolución
- Revisar los clústeres en busca de artefactos técnicos

> 🟢 Las secciones **Core** se ven en vivo. 🔎 Las secciones **Extra** son para estudiar después del tutorial.

## 0. Configuración 🟢
Ejecuta primero la celda de abajo (▶ o `Shift + Enter`). En Google Colab instala los paquetes (~1 min);
en tu computadora asume que ya ejecutaste `pip install -r requirements.txt`.

**Opcional:** cambia `USE_DRIVE = True` para guardar tus resultados en Google Drive y reutilizarlos en
los siguientes notebooks. Si lo dejas en `False`, cada notebook reconstruye lo que necesita automáticamente.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = guardar resultados en Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Funciones auxiliares del tutorial (descarga + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # ejecutando desde un clon local
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. Cargar los datos con QC 🟢
`tut.get_checkpoint("qc")` carga el archivo guardado al final del notebook 01. Si no existe (nueva sesión de Colab),
lo reconstruye desde los datos crudos con los mismos parámetros.

In [ ]:
adata = tut.get_checkpoint("qc", WORKDIR)
adata

## 2. PCA: de 2.000 genes a ~30 ejes 🟢
Los genes están correlacionados (una célula T expresa *CD3D*, *CD3E*, *IL7R* juntos), así que unos pocos
**componentes principales** capturan la mayor parte de la señal biológica. Los pasos siguientes usan los PCs en vez
de los genes: más rápido y con menos ruido.

> 🔎 Muchos tutoriales escalan cada gen a varianza unitaria (`sc.pp.scale`) antes del PCA. Las buenas prácticas
> actuales (sc-best-practices.org) a menudo lo omiten; ambas opciones funcionan para PBMCs.

In [ ]:
sc.tl.pca(adata, n_comps=50, mask_var="highly_variable", random_state=0)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

**Gráfico de codo:** después del "codo" cada PC extra aporta poco. Para PBMC 3k está alrededor del PC 10–15;
conservar algunos más (usamos **30**) es seguro, mientras que conservar muy pocos puede fusionar poblaciones raras.

In [ ]:
sc.pl.pca(adata, color=["CST3", "NKG7", "MS4A1", "pct_counts_mt"], ncols=4, use_raw=True)

PC1 ya separa las células mieloides (*CST3*) de los linfocitos, y el % MT no domina los ejes principales: buena señal.

## 3. Grafo de vecinos y UMAP 🟢
1. **Grafo kNN:** cada célula se conecta con sus 15 células más parecidas en el espacio de PCA. El clustering y el UMAP usan este grafo.
2. **UMAP** dibuja ese grafo en 2D para visualizarlo.

⚠️ UMAP conserva los vecindarios **locales**. Las distancias **entre** clústeres y los **tamaños** de los clústeres en un UMAP
no son confiables: nunca saques conclusiones biológicas solo de ellos.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"], ncols=3)

In [ ]:
sc.pl.umap(adata, color=["CD3E", "CD8A", "GNLY", "MS4A1", "CD14", "FCGR3A", "FCER1A", "PPBP"],
           ncols=4, use_raw=True, cmap="viridis")

## 4. Clustering con Leiden 🟢
Leiden encuentra grupos de células más conectadas entre sí que con el resto del grafo.
La **resolución** controla cuántos clústeres obtienes: más alta = más clústeres, más pequeños.
No existe un único valor correcto; depende de la pregunta biológica.

In [ ]:
for res in [0.3, 0.8, 1.5]:
    sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    print(f"resolution {res}: {adata.obs[f'leiden_{res}'].nunique()} clusters")

sc.pl.umap(adata, color=["leiden_0.3", "leiden_0.8", "leiden_1.5"], legend_loc="on data", ncols=3)

¿Cómo se dividen los clústeres al aumentar la resolución? Una tabla cruzada lo muestra (el paquete de R **clustree** dibuja la misma idea como un árbol).

In [ ]:
pd.crosstab(adata.obs["leiden_0.3"], adata.obs["leiden_0.8"])

Nos quedamos con la **resolución 0,8** como clustering principal (`adata.obs["leiden"]`).

## 5. ¿Hay clústeres que sean artefactos técnicos? 🟢
Un clúster definido solo por pocos UMIs o alto % MT suele ser de células de baja calidad, no un tipo celular.

In [ ]:
adata.obs["leiden"] = adata.obs["leiden_0.8"]
sc.pl.violin(adata, ["n_genes_by_counts", "pct_counts_mt", "doublet_score"], groupby="leiden",
             stripplot=False, multi_panel=True)
adata.obs["leiden"].value_counts().sort_index().to_frame("cells").T

## 6. 🔎 Extra: t-SNE y PAGA
t-SNE es una alternativa más antigua a UMAP. PAGA resume cómo se conectan los clústeres, útil para procesos continuos
(diferenciación) y como mejor inicialización para UMAP.

In [ ]:
sc.tl.tsne(adata, n_pcs=30, random_state=0)
sc.tl.paga(adata, groups="leiden")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sc.pl.tsne(adata, color="leiden", ax=axes[0], show=False, legend_loc="on data")
sc.pl.paga(adata, ax=axes[1], show=False, fontsize=9)
plt.tight_layout()

## 7. Guardar el checkpoint 🟢

In [ ]:
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_clustered.h5ad")

### 💡 Te toca a ti
Recalcula el grafo de vecinos con `n_neighbors=5` y con `n_neighbors=50`, ejecuta UMAP cada vez y compara con el
gráfico de arriba (colorea por `leiden`). ¿Qué cambia? ¿Cambian los clústeres?

In [ ]:
# Tu código aquí

<details><summary><b>Ver solución</b></summary>

```python
for k in (5, 50):
    tmp = adata.copy()
    sc.pp.neighbors(tmp, n_neighbors=k, n_pcs=30, random_state=0)
    sc.tl.umap(tmp, random_state=0)
    sc.pl.umap(tmp, color="leiden", title=f"n_neighbors = {k}", legend_loc="on data")
# k pequeño: estructura local, fragmentada. k grande: más suave, más global.
# Mismos colores = mismos clústeres (calculados antes): UMAP es solo un dibujo del grafo.
```
</details>

---
## ✅ ¡Listo!
Ya tienes clústeres. Pero, ¿qué tipos celulares son? Siguiente: **03 · Anotación de tipos celulares**.